# 02 — Prompt Templates and Mutations

Demonstrates the `PromptTemplate` system and all 10 mutation operators.

In [ ]:
import sys
sys.path.insert(0, '../../')

from src.prompts.template import PromptTemplate
from src.prompts.mutations import PromptMutator

In [ ]:
base = PromptTemplate(task_description="Answer the following question accurately.")

SAMPLE_PROBLEM = "Can a WSU student who has completed MATH 171 and CPTS 121 enroll in CPTS 360?"

print("=== Base Template ===")
print(base.render(SAMPLE_PROBLEM))
print(f"\nMutation path: {base.mutation_path()}")

## Mutation 1: Add Chain-of-Thought

In [ ]:
with_cot = PromptMutator.add_cot(base)
print("=== + add_cot ===")
print(with_cot.render(SAMPLE_PROBLEM))
print(f"Path: {with_cot.mutation_path()}")

## Mutation 2: Add Domain Context (WSU Course Planning)

In [ ]:
with_domain = PromptMutator.add_domain_context(with_cot, 'course_planning')
print("=== + add_domain:course_planning ===")
print(with_domain.render(SAMPLE_PROBLEM))
print(f"Path: {with_domain.mutation_path()}")

## Mutation 3: Add Few-Shot Example

In [ ]:
example = (
    "Q: Can a student with CPTS 121 take CPTS 223?\n"
    "A: Yes. CPTS 223 requires CPTS 121 or CPTS 131. The student has CPTS 121, so they qualify."
)
with_example = PromptMutator.add_example(with_domain, example)
print("=== + add_example ===")
print(with_example.render(SAMPLE_PROBLEM))
print(f"Path: {with_example.mutation_path()}")

## Mutation 4: Set Output Format

In [ ]:
with_format = PromptMutator.set_output_format(with_example, "Yes/No followed by a brief explanation")
print("=== + set_output_format ===")
print(with_format.render(SAMPLE_PROBLEM))
print(f"Path: {with_format.mutation_path()}")

## Mutation 5: Add Constraint

In [ ]:
with_constraint = PromptMutator.add_constraint(base, "Cite the specific prerequisite rule that applies")
print("=== + add_constraint ===")
print(with_constraint.render(SAMPLE_PROBLEM))
print(f"Path: {with_constraint.mutation_path()}")

## Mutation 6: Rephrase Task

In [ ]:
rephrased = PromptMutator.rephrase_task(
    base,
    "Determine whether the student meets the prerequisites for the requested course."
)
print("=== + rephrase_task ===")
print(rephrased.render(SAMPLE_PROBLEM))
print(f"Path: {rephrased.mutation_path()}")

## Mutation 7: Add Verification Step

In [ ]:
with_verify = PromptMutator.add_verification_step(base)
print("=== + add_verification_step ===")
print(with_verify.render(SAMPLE_PROBLEM))
print(f"Path: {with_verify.mutation_path()}")

## Mutation 8: Add Self-Consistency

In [ ]:
with_sc = PromptMutator.add_self_consistency(base)
print("=== + add_self_consistency ===")
print(with_sc.render(SAMPLE_PROBLEM))
print(f"Path: {with_sc.mutation_path()}")

## Mutation 9: Add Expert Persona

In [ ]:
with_persona = PromptMutator.add_expert_persona(base, "a WSU department registrar with 20 years of experience")
print("=== + add_expert_persona ===")
print(with_persona.render(SAMPLE_PROBLEM))
print(f"Path: {with_persona.mutation_path()}")

## Mutation 10: Remove Chain-of-Thought (Ablation)

In [ ]:
without_cot = PromptMutator.remove_cot(with_cot)
print("=== + remove_cot (from with_cot) ===")
print(without_cot.render(SAMPLE_PROBLEM))
print(f"Path: {without_cot.mutation_path()}")

## Full Mutation Chain — Fully Optimized Template

In [ ]:
optimized = base
optimized = PromptMutator.add_domain_context(optimized, 'course_planning')
optimized = PromptMutator.add_cot(optimized)
optimized = PromptMutator.add_example(optimized, example)
optimized = PromptMutator.add_verification_step(optimized)
optimized = PromptMutator.set_output_format(optimized, "Yes/No, then reasoning, then sources")

print("=== Fully Optimized Template ===")
print(optimized.render(SAMPLE_PROBLEM))
print(f"\nMutation path: {optimized.mutation_path()}")

## Mutation Space Size

In [ ]:
parameterized_mutations = [
    ('add_cot', lambda t: PromptMutator.add_cot(t)),
    ('remove_cot', lambda t: PromptMutator.remove_cot(t)),
    ('add_domain:course_planning', lambda t: PromptMutator.add_domain_context(t, 'course_planning')),
    ('add_domain:math_reasoning', lambda t: PromptMutator.add_domain_context(t, 'math_reasoning')),
    ('add_verification', lambda t: PromptMutator.add_verification_step(t)),
    ('add_self_consistency', lambda t: PromptMutator.add_self_consistency(t)),
    ('add_expert:registrar', lambda t: PromptMutator.add_expert_persona(t, 'a WSU registrar')),
    ('add_expert:advisor', lambda t: PromptMutator.add_expert_persona(t, 'an academic advisor')),
    ('add_example', lambda t: PromptMutator.add_example(t, example)),
    ('set_format:yes_no', lambda t: PromptMutator.set_output_format(t, 'Yes/No with reasoning')),
]

print(f"Single-step mutation candidates from base: {len(parameterized_mutations)}")
for name, fn in parameterized_mutations:
    mutated = fn(base)
    print(f"  {name}: history={mutated.history}")